# 02 — Baseline Models: ARIMA + Prophet
**Project:** Probabilistic AQI Forecasting  

Goals:
1. Fit ARIMA(1,1,0) and ARIMA(1,1,1) on daily PM2.5 — compare AIC
2. Fit Prophet with yearly + weekly seasonality
3. Evaluate both on the test set: RMSE, MAE, MAPE
4. Plot actuals vs. predictions for visual sanity check

> These are **point forecast** baselines. Their RMSE becomes the bar the probabilistic model must beat.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11
})

CITY    = 'Delhi'
TARGET  = 'PM2.5'
STEM    = f'../data/processed/{CITY.lower()}_aqi'

AQI_THRESHOLDS = {
    'Good': 30, 'Satisfactory': 60,
    'Moderate': 90, 'Poor': 120, 'Very Poor': 250
}

## 1. Load train / val / test splits

In [ ]:
train = pd.read_csv(f'{STEM}_train.csv', index_col='Datetime', parse_dates=True)[TARGET]
val   = pd.read_csv(f'{STEM}_val.csv',   index_col='Datetime', parse_dates=True)[TARGET]
test  = pd.read_csv(f'{STEM}_test.csv',  index_col='Datetime', parse_dates=True)[TARGET]

print(f'Train : {len(train):,} rows  ({train.index.min().date()} → {train.index.max().date()})')
print(f'Val   : {len(val):,} rows  ({val.index.min().date()} → {val.index.max().date()})')
print(f'Test  : {len(test):,} rows  ({test.index.min().date()} → {test.index.max().date()})')

# ARIMA works on daily data to keep runtime manageable
train_d = train.resample('D').mean().dropna()
val_d   = val.resample('D').mean().dropna()
test_d  = test.resample('D').mean().dropna()
print(f'\nDaily train: {len(train_d)} days | val: {len(val_d)} | test: {len(test_d)}')

## 2. Confirm stationarity on training data

In [ ]:
def adf_report(series, label):
    stat, p, lags, *_ = adfuller(series.dropna(), autolag='AIC')
    result = 'STATIONARY ✓' if p < 0.05 else 'NON-STATIONARY — difference before ARIMA'
    print(f'[{label}]  ADF={stat:.3f}  p={p:.5f}  →  {result}')
    return p

p_level = adf_report(train_d,          'Level     ')
p_diff  = adf_report(train_d.diff(),   'Diff(d=1) ')

## 3. ARIMA — order selection via AIC

In [ ]:
# From EDA: PACF cuts off at lag 1 → p=1; slow ACF decay → d=1
# Compare two candidate orders

candidates = [(1, 1, 0), (1, 1, 1), (2, 1, 1)]
aic_results = []

for order in candidates:
    try:
        m = ARIMA(train_d, order=order).fit()
        aic_results.append((order, round(m.aic, 2)))
        print(f'ARIMA{order}  AIC = {m.aic:.2f}')
    except Exception as e:
        print(f'ARIMA{order}  failed: {e}')

best_order = min(aic_results, key=lambda x: x[1])[0]
print(f'\n→ Best order by AIC: ARIMA{best_order}')

## 4. ARIMA — fit on train, forecast on test

We use a **rolling (walk-forward) forecast**: refit or extend the model one step at a time as new actuals arrive. This is the honest evaluation strategy — it mimics production use.

In [ ]:
def arima_rolling_forecast(train_series, test_series, order):
    """
    Walk-forward forecast: predict one day at a time,
    appending the true value before the next prediction.
    """
    history = list(train_series)
    preds   = []

    for t in range(len(test_series)):
        model = ARIMA(history, order=order)
        fit   = model.fit()
        yhat  = fit.forecast(steps=1)[0]
        preds.append(yhat)
        history.append(test_series.iloc[t])   # reveal true value

        if t % 30 == 0:
            print(f'  step {t}/{len(test_series)}', end='\r')

    return pd.Series(preds, index=test_series.index)


print(f'Running ARIMA{best_order} walk-forward forecast...')
# Use train + val as history for final test evaluation
full_train_d = pd.concat([train_d, val_d])
arima_preds  = arima_rolling_forecast(full_train_d, test_d, best_order)
print('\nDone.')

## 5. Prophet — fit and forecast

In [ ]:
def fit_prophet(train_series, val_series):
    """
    Prophet expects a DataFrame with columns 'ds' (date) and 'y' (value).
    We fit on train+val, then forecast over the test period.
    """
    history = pd.concat([train_series, val_series])
    df_train = pd.DataFrame({
        'ds': history.index,
        'y' : history.values
    })

    model = Prophet(
        yearly_seasonality  = True,
        weekly_seasonality  = True,
        daily_seasonality   = False,   # daily granularity here
        seasonality_mode    = 'multiplicative',   # better for skewed data
        changepoint_prior_scale = 0.1,            # slight regularisation
        interval_width      = 0.80                # 80% uncertainty interval
    )
    model.fit(df_train)
    return model


print('Fitting Prophet...')
prophet_model = fit_prophet(train_d, val_d)

# Build future DataFrame for the test period
future = pd.DataFrame({'ds': test_d.index})
prophet_forecast = prophet_model.predict(future)

prophet_preds = pd.Series(
    prophet_forecast['yhat'].values,
    index=test_d.index
).clip(lower=0)   # PM2.5 can't be negative

print('Done.')

## 6. Evaluation metrics

In [ ]:
def evaluate(actuals, predictions, model_name):
    """Return RMSE, MAE, MAPE for a point forecast."""
    # Align index (safety)
    idx   = actuals.index.intersection(predictions.index)
    y     = actuals[idx].values
    yhat  = predictions[idx].values

    rmse  = np.sqrt(mean_squared_error(y, yhat))
    mae   = mean_absolute_error(y, yhat)
    # MAPE — exclude near-zero actuals to avoid division issues
    mask  = y > 5
    mape  = np.mean(np.abs((y[mask] - yhat[mask]) / y[mask])) * 100

    print(f'{model_name:<20}  RMSE={rmse:6.2f}  MAE={mae:6.2f}  MAPE={mape:5.1f}%')
    return {'model': model_name, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape}


print(f'{'Model':<20}  {'RMSE':>8}  {'MAE':>7}  {'MAPE':>7}')
print('─' * 50)

results = []
results.append(evaluate(test_d, arima_preds,   f'ARIMA{best_order}'))
results.append(evaluate(test_d, prophet_preds, 'Prophet'))

results_df = pd.DataFrame(results).set_index('model')
results_df

## 7. Visual comparison

In [ ]:
# Plot last 90 days of test set for readability
window = test_d.iloc[-90:]

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

for ax, (preds, label, color) in zip(axes, [
    (arima_preds,   f'ARIMA{best_order}', '#e07b54'),
    (prophet_preds, 'Prophet',            '#5b9bd5'),
]):
    ax.plot(window.index, window.values,
            color='#333', lw=1.2, label='Actual', zorder=3)
    ax.plot(window.index, preds[window.index],
            color=color, lw=1.2, ls='--', label=label, zorder=2)

    for name, thresh in AQI_THRESHOLDS.items():
        ax.axhline(thresh, lw=0.6, ls=':', color='tomato', alpha=0.5)
        ax.text(window.index[-1], thresh + 2, name, fontsize=7, color='tomato')

    rmse = results_df.loc[label, 'RMSE'] if label in results_df.index \
           else results_df.loc[f'ARIMA{best_order}', 'RMSE']
    ax.set_title(f'{label}  (RMSE = {results_df.loc[label if label in results_df.index else f"ARIMA{best_order}", "RMSE"]:.2f}  µg/m³)')
    ax.set_ylabel('PM2.5 (µg/m³)')
    ax.legend(loc='upper left', fontsize=9)

plt.suptitle(f'Baseline forecasts — {CITY} (last 90 test days)', y=1.01)
plt.tight_layout()
plt.savefig('../visualizations/02_baseline_forecasts.png', bbox_inches='tight')
plt.show()

## 8. Prophet components

Prophet's built-in component plot shows trend, yearly seasonality, and weekly seasonality separately. Useful to confirm it captured the winter/summer cycle.

In [ ]:
fig = prophet_model.plot_components(prophet_forecast)
plt.suptitle('Prophet components', y=1.01)
plt.tight_layout()
plt.savefig('../visualizations/02_prophet_components.png', bbox_inches='tight')
plt.show()

## 9. Save results for comparison table

In [ ]:
results_df.to_csv('../evaluation/baseline_results.csv')

# Save predictions for later plotting alongside LSTM / Quantile model
arima_preds.rename('arima').to_csv('../evaluation/arima_test_preds.csv')
prophet_preds.rename('prophet').to_csv('../evaluation/prophet_test_preds.csv')

print('Saved to evaluation/')
print()
print('── Baseline summary ─────────────────────────────')
print(results_df.round(2).to_string())
print()
print('These are your baseline numbers.')
print('The LSTM (notebook 03) should beat both on RMSE.')
print('The Quantile LSTM (notebook 04) adds uncertainty on top.')